Нужно предугадать тип-источник запаха

In [1]:
import pandas as pd
import random

In [2]:
FILENAME = '../results/Разметка_сравнение_RU_EN.xlsx'

In [ ]:
# Загружаем ваши данные
df = pd.read_excel(FILENAME, sheet_name='Качество', header=0)

# Выбираем 100 предложений для разметки
sample_df = df[['sent_text_RU']].copy()
sample_df = sample_df.sample(n=100, random_state=42)
sample_df['source'] = ''

# Сохраняем для разметки
sample_df.to_csv('to_annotate.csv', index=False, encoding='utf-8-sig')

print(f"✅ 100 предложений сохранены в to_annotate.csv")
print("Откройте файл в Excel и добавьте колонку 'source_type'")
print("Варианты: plant, food, chemical, animal, other")

✅ 100 предложений сохранены в to_annotate.csv
Откройте файл в Excel и добавьте колонку 'source_type'
Варианты: plant, food, chemical, animal, other


### train_smell_classifier.py

In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [12]:
# Загружаем данные
df = pd.read_csv('to_annotate.csv')
df['source'] = df['source'].fillna('other')
# Подготовка данных
X = df['sent_text_RU']  # текст
y = df['source']   # тип источника

# Превращаем текст в числа (TF-IDF)
vectorizer = TfidfVectorizer(
    max_features=1000,      # топ-1000 слов
    ngram_range=(1, 2),     # учитываем пары слов
    min_df=2,               # слово должно быть минимум в 2 предложениях
    stop_words=['пахло', 'запах', 'пахнет']  # убираем слишком частые слова
)

X_vectorized = vectorizer.fit_transform(X)

# Разделяем на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2, random_state=42
)

print(f"\nОбучающая выборка: {X_train.shape[0]} предложений")
print(f"Тестовая выборка: {X_test.shape[0]} предложений")



Обучающая выборка: 80 предложений
Тестовая выборка: 20 предложений


In [14]:

# ============================================
# МОДЕЛЬ 1: Логистическая регрессия (вы знаете)
# ============================================
print("\n" + "="*50)
print("ОБУЧЕНИЕ ЛОГИСТИЧЕСКОЙ РЕГРЕССИИ")
print("="*50)

lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,                  # сила регуляризации     
    random_state=42
)
lr_model.fit(X_train, y_train)

# Оценка
lr_score = cross_val_score(lr_model, X_vectorized, y, cv=5)
print(f"Cross-validation accuracy: {lr_score.mean():.3f} (+/- {lr_score.std():.3f})")
print(f"Тестовая точность: {lr_model.score(X_test, y_test):.3f}")


ОБУЧЕНИЕ ЛОГИСТИЧЕСКОЙ РЕГРЕССИИ
Cross-validation accuracy: 0.440 (+/- 0.020)
Тестовая точность: 0.550


d:\Python\olf\.venv2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [15]:

# ============================================
# МОДЕЛЬ 2: Random Forest (вы знаете)
# ============================================
print("\n" + "="*50)
print("ОБУЧЕНИЕ RANDOM FOREST")
print("="*50)

rf_model = RandomForestClassifier(
    n_estimators=100,       # 100 деревьев
    max_depth=10,           # глубина дерева
    random_state=42
)
rf_model.fit(X_train, y_train)

# Оценка
rf_score = cross_val_score(rf_model, X_vectorized, y, cv=5)
print(f"Cross-validation accuracy: {rf_score.mean():.3f} (+/- {rf_score.std():.3f})")
print(f"Тестовая точность: {rf_model.score(X_test, y_test):.3f}")


ОБУЧЕНИЕ RANDOM FOREST


d:\Python\olf\.venv2\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Cross-validation accuracy: 0.430 (+/- 0.051)
Тестовая точность: 0.600


In [16]:
# ============================================
# СРАВНЕНИЕ МОДЕЛЕЙ
# ============================================
print("\n" + "="*50)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*50)

print(f"Логистическая регрессия: {lr_score.mean():.3f}")
print(f"Random Forest: {rf_score.mean():.3f}")

# Выбираем лучшую модель
if lr_score.mean() > rf_score.mean():
    best_model = lr_model
    best_name = "Logistic Regression"
else:
    best_model = rf_model
    best_name = "Random Forest"

print(f"\n✅ Лучшая модель: {best_name}")


СРАВНЕНИЕ РЕЗУЛЬТАТОВ
Логистическая регрессия: 0.440
Random Forest: 0.430

✅ Лучшая модель: Logistic Regression


In [17]:
# ============================================
# АНАЛИЗ ОШИБОК
# ============================================
print("\n" + "="*50)
print("ДЕТАЛЬНЫЙ АНАЛИЗ")
print("="*50)

# Предсказания на тесте
y_pred = best_model.predict(X_test)

# Отчет по каждому классу
print("\nКлассификационный отчет:")
print(classification_report(y_test, y_pred))

# Матрица ошибок
print("\nМатрица ошибок:")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, 
                   index=sorted(y.unique()),
                   columns=sorted(y.unique())))



ДЕТАЛЬНЫЙ АНАЛИЗ

Классификационный отчет:
              precision    recall  f1-score   support

       other       0.53      1.00      0.69        10
        люди       0.00      0.00      0.00         1
   окружение       0.00      0.00      0.00         1
    предметы       0.00      0.00      0.00         1
пространство       1.00      0.20      0.33         5
        тело       0.00      0.00      0.00         2

    accuracy                           0.55        20
   macro avg       0.25      0.20      0.17        20
weighted avg       0.51      0.55      0.43        20


Матрица ошибок:


d:\Python\olf\.venv2\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Python\olf\.venv2\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Python\olf\.venv2\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


ValueError: Shape of passed values is (6, 6), indices imply (15, 15)

In [18]:
# ============================================
# САМЫЕ ВАЖНЫЕ СЛОВА ДЛЯ КАЖДОГО ТИПА
# ============================================
print("\n" + "="*50)
print("САМЫЕ ВАЖНЫЕ СЛОВА ДЛЯ КАЖДОГО ТИПА")
print("="*50)

feature_names = vectorizer.get_feature_names_out()

if best_name == "Logistic Regression":
    coefficients = best_model.coef_
    for i, class_name in enumerate(best_model.classes_):
        top_features = np.argsort(coefficients[i])[-5:]
        top_words = [feature_names[j] for j in top_features]
        print(f"\n{class_name.upper()}:")
        print(f"  Ключевые слова: {', '.join(top_words)}")
else:  # Random Forest
    importances = best_model.feature_importances_
    top_indices = np.argsort(importances)[-10:]
    top_words = [feature_names[i] for i in top_indices]
    print(f"\nТоп-10 самых важных слов:")
    for word in top_words:
        print(f"  - {word}")




САМЫЕ ВАЖНЫЕ СЛОВА ДЛЯ КАЖДОГО ТИПА

OTHER:
  Ключевые слова: запаха, но, запахом, как, на

ЕДА:
  Ключевые слова: были, запахе, еще, пахла, две

ЖИВОТНОЕ:
  Ключевые слова: сердце, от, его, удивительный, шел

ЛЮДИ:
  Ключевые слова: они, них, от них, от, собой

ОДЕЖДА:
  Ключевые слова: три, сердце, то, были, их

ОКРУЖЕНИЕ:
  Ключевые слова: был, было, когда, комнате, удивительный

ПРЕДМЕТ:
  Ключевые слова: то, сердце, роз, тончайший, вокруг

ПРЕДМЕТЫ:
  Ключевые слова: где, были, три, сердце, то

ПРИРОДА:
  Ключевые слова: то, сердце, окна, проникал, теплой

ПРОСТРАНСТВО:
  Ключевые слова: сада, пылью, душных, травой, навозом

РАСТЕНИЕ:
  Ключевые слова: пахли, сильно, сладко, водорослей, от

СИТУАЦИЯ:
  Ключевые слова: казалось, тогда, пахнуть, иногда, вот

ТЕЛО:
  Ключевые слова: вас, пахнут, от нее, нее, от

ЦВЕТЫ:
  Ключевые слова: три, их, даже, проникал, распространяя

ЧЕЛОВЕК:
  Ключевые слова: сердце, от, него, от него, дымом


In [ ]:
# ============================================
# СОХРАНЯЕМ МОДЕЛЬ
# ============================================
print("\n" + "="*50)
print("СОХРАНЕНИЕ МОДЕЛИ")
print("="=50)

joblib.dump(best_model, 'smell_classifier_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("✅ Модель сохранена в 'smell_classifier_model.pkl'")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sentence_transformers import SentenceTransformer

# Загружаем данные
df = pd.read_csv('to_annotate.csv')

# ============================================
# ШАГ 1: УПРОЩАЕМ ЗАДАЧУ
# ============================================
# Создаем бинарную классификацию (natural vs artificial)
df['binary_type'] = df['source'].apply(lambda x: 
    'natural' if x in ['plant', 'food', 'animal', 'природа', 'еда'] 
    else 'artificial'
)

print("Распределение классов:")
print(df['binary_type'].value_counts())
print(f"Всего примеров: {len(df)}")

# ============================================
# ШАГ 2: ПРИЗНАКИ (TF-IDF + эмбеддинги)
# ============================================
# TF-IDF
vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(df['sent_text_RU'])

# Эмбеддинги
print("Загрузка эмбеддингов...")
encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = encoder.encode(df['sent_text_RU'].tolist(), show_progress_bar=True)

# Объединяем
X = np.hstack([X_tfidf.toarray(), embeddings])
y = df['binary_type']

# ============================================
# ШАГ 3: ОБУЧЕНИЕ
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8),
    # 'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
}

print("\n" + "="*50)
print("СРАВНЕНИЕ МОДЕЛЕЙ (natural vs artificial)")
print("="*50)

for name, model in models.items():
    # Кросс-валидация
    cv_scores = cross_val_score(model, X_train, y_train, cv=min(5, len(y_train)-1))
    print(f"\n{name}:")
    print(f"  Cross-val: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
    
    # Обучаем и тестируем
    model.fit(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    print(f"  Тест: {test_acc:.3f}")

# ============================================
# ШАГ 4: АНАЛИЗ ОШИБОК
# ============================================
best_model = LogisticRegression(max_iter=1000)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("\n" + "="*50)
print("АНАЛИЗ ОШИБОК")
print("="*50)

# Какие предложения модель путает?
errors = X_test[y_test != y_pred]
error_texts = df.iloc[X_test.index[y_test != y_pred]]['sent_text_RU']

print("\nОшибки модели (неправильно классифицировала):")
for i, (text, true, pred) in enumerate(zip(
    error_texts[:10], 
    y_test[y_test != y_pred][:10], 
    y_pred[y_test != y_pred][:10]
)):
    print(f"{i+1}. '{text}'")
    print(f"   Правильно: {true}, Модель: {pred}")

# ============================================
# ШАГ 5: ВЫВОДЫ ДЛЯ ДИССЕРТАЦИИ
# ============================================
print("\n" + "="*50)
print("РЕКОМЕНДАЦИИ ДЛЯ УЛУЧШЕНИЯ")
print("="*50)

print("""
1. Соберите больше данных (сейчас мало)
2. Упростите классификацию (natural vs artificial работает лучше)
3. Добавьте лемматизацию (сейчас модель не видит связи 'пахло' и 'пахнет')
4. Используйте кросс-валидацию для надежной оценки
""")

Распределение классов:
binary_type
artificial    95
natural        5
Name: count, dtype: int64
Всего примеров: 100
Загрузка эмбеддингов...


d:\Python\olf\.venv2\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lolla\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
